In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session



/kaggle/input/learning-agency-lab-automated-essay-scoring-2/sample_submission.csv
/kaggle/input/learning-agency-lab-automated-essay-scoring-2/train.csv
/kaggle/input/learning-agency-lab-automated-essay-scoring-2/test.csv


In [2]:
!pwd

/kaggle/working


In [3]:
train_df = pd.read_csv('../input/learning-agency-lab-automated-essay-scoring-2/train.csv', index_col = 'essay_id')

In [4]:
test_df = pd.read_csv('../input/learning-agency-lab-automated-essay-scoring-2/test.csv', index_col = 'essay_id')

In [5]:
submission_df = pd.read_csv('../input/learning-agency-lab-automated-essay-scoring-2/sample_submission.csv')

In [6]:
# word2vecを用いて、単語をベクトル化するためのデータクレンジングをします
import re
import subprocess


from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA





In [7]:
# 大文字を小文字にそろえる

train_df.full_text = train_df.full_text.str.lower()
test_df.full_text = test_df.full_text.str.lower()

# テキストの前処理
# 1. 数字を0に置換

# train_df.full_text = train_df.full_text.str.replace(r'[0-9]+', '0', regex=True)

#　test_df.full_text = test_df.full_text.str.replace(r'[0-9]+', '0', regex=True)

# 2. 記号を削除

train_df.full_text = train_df.full_text.str.replace(r'[!-/:-@[-`{-~]', ' ', regex=True)

test_df.full_text = test_df.full_text.str.replace(r'[!-/:-@[-`{-~]', ' ', regex=True)

# 3. 余分な空白を削除

train_df.full_text = train_df.full_text.str.replace(r'\s+', ' ', regex=True)

test_df.full_text = test_df.full_text.str.replace(r'\s+', ' ', regex=True)


In [8]:
# word2vecを用いて、単語をベクトル化します

word2vec_model = Word2Vec(train_df['full_text'], 
                          vector_size=100,
                          min_count=1,
                          window=5, 
                          )

In [9]:
def vectorize_text(text):
    text_vector = np.zeros(100)
    for word in text:
        try:
            text_vector += word2vec_model.wv[word]
        except:
            pass
    return text_vector


In [10]:
train_df['text_vector'] = train_df['full_text'].apply(vectorize_text)

train_df.shape

(17307, 3)

In [11]:
'''
# PCAを用いて、ベクトル化した単語を次元削減します

pca = PCA(n_components=10)

train_df['pca_vector'] = list(pca.fit_transform(train_df['text_vector'].to_list()))



test_df['pca_vector'] = list(pca.transform(test_df['text_vector'].to_list()))

'''
test_df['text_vector'] = test_df['full_text'].apply(vectorize_text)


In [12]:
train_df.shape

(17307, 3)

In [13]:
# lightgbmを用いてモデルを作ります

import lightgbm as lgb
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import train_test_split

X = np.array(train_df['text_vector'].to_list())

y = train_df['score']


In [14]:
# yが１～６なので、０～５に入れ替えます

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_ = le.fit_transform(y)

In [15]:
# データをlightgbm用に変換します

lgb_train = lgb.Dataset(X, y_)


In [16]:
#　cohen_kappa_scoreを評価関数として、lightgbmを用いて6クラス分類のモデルを作ります

params = {
    'objective': 'multiclass',
    'num_class': 6,
    'metric': 'multi_logloss',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9,
}

In [17]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

In [18]:
lgb_valid = lgb.Dataset(X_valid, y_valid, reference=lgb_train)

In [19]:
model = lgb.train(params, lgb_train, valid_sets=lgb_valid, num_boost_round=1000) 

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.017573 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 25500
[LightGBM] [Info] Number of data points in the train set: 17307, number of used features: 100
[LightGBM] [Info] Start training from score -2.626369
[LightGBM] [Info] Start training from score -1.298667
[LightGBM] [Info] Start training from score -1.013741
[LightGBM] [Info] Start training from score -1.483490
[LightGBM] [Info] Start training from score -2.881570
[LightGBM] [Info] Start training from score -4.709010
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM]

In [20]:
# 学習をするためにlightBGMの形式でテストデータを作ります。

X_test = np.array(test_df['text_vector'].to_list())

In [21]:
y_pred = model.predict(X_test)


In [22]:
y_pred = np.argmax(y_pred, axis=1)

In [23]:
y_pred

array([2, 2, 3])

In [24]:
prediction = le.inverse_transform(y_pred)

In [25]:
prediction

array([3, 3, 4])

In [26]:
submission_df['score'] = prediction

In [27]:
submission_df.to_csv('submission.csv', index = False)